# 1. Import Framework

In [54]:
#load libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder

# 2. Load Data

In [55]:
#Load the Data
train_transaction = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv")
train_identity    = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv")
test_transaction  = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv")
test_identity     = pd.read_csv("/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv")


# 3. Merge Data

In [56]:
#Merge the Data
train = train_transaction.merge(train_identity, on="TransactionID", how="left")
test  = test_transaction.merge(test_identity, on="TransactionID", how="left")

# 4. Memory Optimizatio

In [57]:
def reduce_mem(df):
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
        else:
            df[col] = df[col].astype("category")
    return df

train = reduce_mem(train)
test  = reduce_mem(test)


# 5. Split Data into Features and Target

In [58]:
#Separate Target and Features
y = train["isFraud"]
X = train.drop("isFraud", axis=1)
X_test = test.copy()


# 6. Align Train And Test Columns

In [59]:
#Align columns between train and test
missing_cols = [col for col in X.columns if col not in X_test.columns]
X_test = pd.concat([X_test, pd.DataFrame(np.nan, index=X_test.index, columns=missing_cols)], axis=1)
X_test = X_test[X.columns]


# 7. Feature Engineering 

In [60]:
# --- Time Features ---
X["day"] = X["TransactionDT"] // (24*60*60)
X_test["day"] = X_test["TransactionDT"] // (24*60*60)

X["hour"] = (X["TransactionDT"] // 3600) % 24
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24

# --- Amount Features ---
X["log_amt"] = np.log1p(X["TransactionAmt"])
X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])

X["amt_per_day"] = X["TransactionAmt"] / (X["day"] + 1)
X_test["amt_per_day"] = X_test["TransactionAmt"] / (X_test["day"] + 1)

# --- Frequency Encoding ---
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]

for col in freq_cols:
    freq = X[col].value_counts()
    X[col + "_freq"] = X[col].map(freq)
    X_test[col + "_freq"] = X_test[col].map(freq)

# --- Device Features ---# --- Time Features ---
X["day"] = X["TransactionDT"] // (24*60*60)
X_test["day"] = X_test["TransactionDT"] // (24*60*60)

X["hour"] = (X["TransactionDT"] // 3600) % 24
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24

# --- Amount Features ---
X["log_amt"] = np.log1p(X["TransactionAmt"])
X_test["log_amt"] = np.log1p(X_test["TransactionAmt"])

X["amt_per_day"] = X["TransactionAmt"] / (X["day"] + 1)
X_test["amt_per_day"] = X_test["TransactionAmt"] / (X_test["day"] + 1)

# --- Frequency Encoding ---
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]

for col in freq_cols:
    freq = X[col].value_counts()
X["device_name"] = X["DeviceInfo"].str.split("/", expand=True)[0]
X_test["device_name"] = X_test["DeviceInfo"].str.split("/", expand=True)[0]

X["device_version"] = X["DeviceInfo"].str.split("/", expand=True)[1]
X_test["device_version"] = X_test["DeviceInfo"].str.split("/", expand=True)[1]

# --- Email Domain Grouping ---
email_map = {
    'gmail.com': 'google', 'gmail': 'google',
    'yahoo.com': 'yahoo', 'yahoo.com.mx': 'yahoo',
    'hotmail.com': 'microsoft', 'outlook.com': 'microsoft'
}

X["P_emaildomain_group"] = X["P_emaildomain"].map(email_map)
X_test["P_emaildomain_group"] = X_test["P_emaildomain"].map(email_map)


/tmp/ipykernel_58/2460950459.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["day"] = X["TransactionDT"] // (24*60*60)
/tmp/ipykernel_58/2460950459.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X["hour"] = (X["TransactionDT"] // 3600) % 24
/tmp/ipykernel_58/2460950459.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, 

In [74]:
# Collect new features in a dict
new_features = {}

# Time features
new_features["day"] = X["TransactionDT"] // (24*60*60)
new_features["hour"] = (X["TransactionDT"] // 3600) % 24

# Amount features
new_features["log_amt"] = np.log1p(X["TransactionAmt"])
new_features["amt_per_day"] = X["TransactionAmt"] / (new_features["day"] + 1)

# Frequency encoding
freq_cols = ["card1", "card2", "addr1", "addr2", "P_emaildomain", "R_emaildomain"]
for col in freq_cols:
    freq = X[col].value_counts()
    new_features[col + "_freq"] = X[col].map(freq)

# Device features
new_features["device_name"] = X["DeviceInfo"].str.split("/", expand=True)[0]
new_features["device_version"] = X["DeviceInfo"].str.split("/", expand=True)[1]

# Email grouping
email_map = {
    'gmail.com': 'google', 'gmail': 'google',
    'yahoo.com': 'yahoo', 'yahoo.com.mx': 'yahoo',
    'hotmail.com': 'microsoft', 'outlook.com': 'microsoft'
}
new_features["P_emaildomain_group"] = X["P_emaildomain"].map(email_map)

# Add all new features at once
X = pd.concat([X, pd.DataFrame(new_features)], axis=1)


AttributeError: Can only use .str accessor with string values!

In [61]:
#Feature Engineering time features
X["day"] = X["TransactionDT"] // (24*60*60)
X["hour"] = (X["TransactionDT"] // 3600) % 24

X_test["day"] = X_test["TransactionDT"] // (24*60*60)
X_test["hour"] = (X_test["TransactionDT"] // 3600) % 24


In [ ]:
# Repeat for X_test
new_features_test = {}

new_features_test["day"] = X_test["TransactionDT"] // (24*60*60)
new_features_test["hour"] = (X_test["TransactionDT"] // 3600) % 24
new_features_test["log_amt"] = np.log1p(X_test["TransactionAmt"])
new_features_test["amt_per_day"] = X_test["TransactionAmt"] / (new_features_test["day"] + 1)

for col in freq_cols:
    freq = X[col].value_counts()
    new_features_test[col + "_freq"] = X_test[col].map(freq)

new_features_test["device_name"] = X_test["DeviceInfo"].str.split("/", expand=True)[0]
new_features_test["device_version"] = X_test["DeviceInfo"].str.split("/", expand=True)[1]
new_features_test["P_emaildomain_group"] = X_test["P_emaildomain"].map(email_map)

X_test = pd.concat([X_test, pd.DataFrame(new_features_test)], axis=1)


# 8. Encode categorical features

In [62]:

categorical_cols = X.select_dtypes(include=["object", "category"]).columns

for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))



# 9. Automatic Column Detection

In [63]:
# Identify numeric and categorical columns:
numeric_cols = X.select_dtypes(include=["int", "float", "float32", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "category"]).columns


# 10. Convert Numeric Columns To Float32

In [64]:
X[numeric_cols] = X[numeric_cols].astype("float32")
X_test[numeric_cols] = X_test[numeric_cols].astype("float32")


# 11. Handle missing values

In [65]:
#Handle Missing Values
num_cols = X.select_dtypes(include=["int", "float"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

X[num_cols] = X[num_cols].fillna(-1)
X_test[num_cols] = X_test[num_cols].fillna(-1)

X[cat_cols] = X[cat_cols].fillna("missing")
X_test[cat_cols] = X_test[cat_cols].fillna("missing")


# 12.Fill Missing Values

In [66]:
X = X.fillna(-1)
X_test = X_test.fillna(-1)


# 13. Train And Validation Split

In [67]:
#train/validation split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# 14. Lightgbm Datasets

In [68]:
train_set = lgb.Dataset(X_train, y_train)
valid_set = lgb.Dataset(X_valid, y_valid)


# 15 Lightgbm Parameters

In [69]:
params = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "learning_rate": 0.01,
    "num_leaves": 256,
    "max_depth": -1,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "lambda_l1": 0.1,
    "lambda_l2": 0.1,
    "verbose": -1
}


# 16 Train The Model

In [70]:
model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=20000,
    callbacks=[
        lgb.early_stopping(500),
        lgb.log_evaluation(100)   # optional: prints every 100 rounds
    ]
)


Training until validation scores don't improve for 500 rounds
[100]	training's auc: 0.932299	valid_1's auc: 0.885346
[200]	training's auc: 0.956126	valid_1's auc: 0.901757
[300]	training's auc: 0.972676	valid_1's auc: 0.91304
[400]	training's auc: 0.983163	valid_1's auc: 0.9206
[500]	training's auc: 0.989373	valid_1's auc: 0.924954
[600]	training's auc: 0.99323	valid_1's auc: 0.926716
[700]	training's auc: 0.995607	valid_1's auc: 0.927973
[800]	training's auc: 0.997175	valid_1's auc: 0.929012
[900]	training's auc: 0.998134	valid_1's auc: 0.929506
[1000]	training's auc: 0.998818	valid_1's auc: 0.929548
[1100]	training's auc: 0.999208	valid_1's auc: 0.92981
[1200]	training's auc: 0.999462	valid_1's auc: 0.929754
[1300]	training's auc: 0.99965	valid_1's auc: 0.92996
[1400]	training's auc: 0.999764	valid_1's auc: 0.930135
[1500]	training's auc: 0.999844	valid_1's auc: 0.93009
[1600]	training's auc: 0.9999	valid_1's auc: 0.929917
[1700]	training's auc: 0.999934	valid_1's auc: 0.929998
[1800


# 17. Validation Predictions and AUC

In [71]:
valid_pred = model.predict(X_valid)
auc = roc_auc_score(y_valid, valid_pred)
print("Validation AUC:", auc)


Validation AUC: 0.9303192689546266


# 18. Test Predictions

In [72]:
test_pred = model.predict(X_test)

# 19. Create Submission File

In [75]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"],
    "isFraud": test_pred
})

submission.to_csv("submission.csv", index=False)
